# Exercise: Getting Comfortable with Our Open Lakehouse

This first exercise is setup to make sure that your container environment correctly started up and that
Apache Iceberg tables are accessible from multiple engines.

## The Tools

- **Apache Iceberg**: Open table format
- **MinIO**: S3-compatible object storage
- **Polaris**: Iceberg REST Compatible Catalog
- **Spark**: Distributed Analytics Engine
- **Trino**: Distributed Analytics Engine

## Explore the Environment


### Jupyter Notebooks

**You're here!**

Jupyter provides an interactive environment for running Spark and Python code. All the lesson exercises are stored in the `/notebooks` directory.

**What to explore:**
- The file browser on the left shows all available notebooks
- Each notebook corresponds to an exercise in the course
- You can create new notebooks to experiment with your own queries or code
- The integrated terminal allows you to run shell commands if needed

### Polaris Catalog

**URL:** Polaris UI is not yet released

Polaris is a REST catalog that manages all the metadata for your Iceberg tables. It keeps track of schemas, snapshots, and table locations without storing the actual data.

**What to explore:**
- TBD

### Spark Configuration

**Configuration File:** `/home/jovyan/.sparkconf/spark-defaults.conf`

The Spark Session configuration is automatically generated by docker-compose and is placed in the above file to wire together all of the components
of this Lakehouse as well as download necessary dependencies for Spark to work with Iceberg. The code below opens that file, reads it into the config variable, and then prints that variable for you to see.

**Key packages included:**
- **Iceberg Spark Runtime** - Iceberg Plugin Integration and Iceberg REST Catalog Client
- **AWS Bundle** - Provides S3 Client for MinIO
- **Catalog Configuration** - Points Spark to the Polaris REST catalog
- **S3 Credentials** - Authenticates with MinIO storage

In [6]:
with open('/home/jovyan/.sparkconf/spark-defaults.conf', 'r') as f:
    config = f.read()
    print(config)

# Iceberg Spark Dependencies
spark.jars.packages=org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.0,org.apache.iceberg:iceberg-aws-bundle:1.10.0,software.amazon.awssdk:bundle:2.20.18,software.amazon.awssdk:url-connection-client:2.20.18,org.apache.hadoop:hadoop-aws:3.4.1

# Iceberg Spark SQL/Catalyst Extensions
spark.sql.extensions=org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions

# Spark Memory Configuration
spark.driver.memory=4g
spark.driver.maxResultSize=2g

# Catalog configuration for Polaris with OAuth2
spark.sql.catalog.polaris=org.apache.iceberg.spark.SparkCatalog
spark.sql.catalog.polaris.type=rest
spark.sql.catalog.polaris.uri=http://polaris:8181/api/catalog
spark.sql.catalog.polaris.warehouse=quickstart_catalog
spark.sql.catalog.polaris.credential=root:s3cr3t
spark.sql.catalog.polaris.scope=PRINCIPAL_ROLE:ALL
spark.sql.catalog.polaris.oauth2-server-uri=http://polaris:8181/api/catalog/v1/oauth/tokens
spark.sql.catalog.polaris.header.X-Iceberg-Access-Dele

## Initialize Spark Session

Now let's start coding! We'll create a Spark session that connects to our pre-configured Iceberg environment. As you can see in the printed result, we are using the Polaris catalog.

In [7]:
from pyspark.sql import SparkSession

# All configuration is loaded from spark-defaults.conf
spark = SparkSession.builder \
    .appName("OpenLakehouse") \
    .getOrCreate()

print(f" Spark {spark.version} initialized!")
print(f" Default Catalog: {spark.conf.get('spark.sql.defaultCatalog', 'spark_catalog')}")

 Spark 4.0.1 initialized!
 Default Catalog: polaris


## Create Your First Iceberg Table

Let's create a simple table to demonstrate the lakehouse in action. We'll create a namespace (like a database schema) and a table within it.

In [8]:


# Create namespace (like a schema in traditional databases)
spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.demo")
print(" Namespace 'demo' created!")

#ERROR: Forbidden: The Access Key Id you provided does not exist in our records.
# Sam still gets this error after starting the docker image
# Create an Iceberg table
spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.demo.employees (
        id INT,
        name STRING,
        department STRING
    ) USING iceberg
    TBLPROPERTIES (
        'format-version' = '3'
    )
""")
print(" Table 'employees' created!") #TODO: Instead of just claiming the table is created, maybe we should do a check that it exists and print the successful check message

 Namespace 'demo' created!


Py4JJavaError: An error occurred while calling o55.sql.
: org.apache.iceberg.exceptions.ForbiddenException: Forbidden: The Access Key Id you provided does not exist in our records. (Service: S3, Status Code: 403, Request ID: 188F9BCB3C965305, Extended Request ID: dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8) (SDK Attempt Count: 1)
	at org.apache.iceberg.rest.ErrorHandlers$DefaultErrorHandler.accept(ErrorHandlers.java:238)
	at org.apache.iceberg.rest.ErrorHandlers$TableErrorHandler.accept(ErrorHandlers.java:124)
	at org.apache.iceberg.rest.ErrorHandlers$TableErrorHandler.accept(ErrorHandlers.java:108)
	at org.apache.iceberg.rest.HTTPClient.throwFailure(HTTPClient.java:240)
	at org.apache.iceberg.rest.HTTPClient.execute(HTTPClient.java:336)
	at org.apache.iceberg.rest.HTTPClient.execute(HTTPClient.java:297)
	at org.apache.iceberg.rest.BaseHTTPClient.get(BaseHTTPClient.java:77)
	at org.apache.iceberg.rest.RESTSessionCatalog.loadInternal(RESTSessionCatalog.java:375)
	at

## Insert Sample Data

Now with our table created and our initial schema set, let's insert a series of data values into this table.

In [9]:
spark.sql("""
    INSERT INTO polaris.demo.employees VALUES
    (1, 'Alice', 'Engineering'),
    (2, 'Bob', 'Sales'),
    (3, 'Charlie', 'Marketing'),
    (4, 'Diana', 'Engineering')
""")
print(" Sample data inserted!")

 Sample data inserted!


## Query with Spark

Now let's confirm that the data is in the table and read the data back using Spark SQL.

In [10]:
spark.sql("""
    SELECT * FROM polaris.demo.employees
""").show()

+---+-------+-----------+
| id|   name| department|
+---+-------+-----------+
|  1|  Alice|Engineering|
|  2|    Bob|      Sales|
|  3|Charlie|  Marketing|
|  4|  Diana|Engineering|
|  1|  Alice|Engineering|
|  2|    Bob|      Sales|
|  3|Charlie|  Marketing|
|  4|  Diana|Engineering|
+---+-------+-----------+



In [6]:
spark.sql("""
    SELECT department, COUNT(*) as employee_count
    FROM polaris.demo.employees
    GROUP BY department
    ORDER BY employee_count DESC
""").show()

+-----------+--------------+
| department|employee_count|
+-----------+--------------+
|Engineering|             2|
|      Sales|             1|
|  Marketing|             1|
+-----------+--------------+



### Explore the Spark UI

**URL:** http://localhost:4040

The Spark UI provides detailed information about your Spark jobs:

**What you can see:**
- **Jobs tab**: Overview of all Spark jobs executed
- **Stages tab**: Breakdown of each job into stages and tasks
- **SQL tab**: Query plans and execution details for Spark SQL operations
- **Environment tab**: Configuration settings and runtime properties

This is useful for understanding query performance and debugging issues.

## Query with Trino

One of the key benefits of Apache Iceberg is **engine interoperability**. Let's connect to Trino and query the exact same data.

Trino is a distributed SQL query engine designed for analytics. It's completely separate from Spark, but both can read/write the same Iceberg tables. Trino is accessible via JDBC which we are using below.

In [11]:
from trino.dbapi import connect
import pandas as pd

# Connect to Trino
conn = connect(
    host='trino',
    port=8080,
    user='trino',
    catalog='polaris',
    schema='demo'
)

cursor = conn.cursor()

# Query the same table we created with Spark!
cursor.execute("SELECT * FROM employees")
rows = cursor.fetchall()
columns = [desc[0] for desc in cursor.description]

# Display results
df = pd.DataFrame(rows, columns=columns)
print(" Querying via Trino:\n")
print(df.to_string(index=False))

cursor.close()
conn.close()

 Querying via Trino:

 id    name  department
  3 Charlie   Marketing
  4   Diana Engineering
  1   Alice Engineering
  2     Bob       Sales
  1   Alice Engineering
  3 Charlie   Marketing
  2     Bob       Sales
  4   Diana Engineering


### Explore the Trino UI

**URL:** http://localhost:8080  
**Credentials:** admin

The Trino UI shows query execution and cluster status:

**What you can see:**
- **Query details**: Click any query to see execution plan, data scanned, and performance metrics. Make sure to select *finished* in the filter to see the above query
- **Cluster overview**: Worker nodes and resource utilization
- **Live query monitoring**: Real-time progress of running queries

The UI is great for monitoring query performance and understanding how Trino processes your SQL.

## Explore MinIO

Now that we've created tables and inserted data, let's explore MinIO to see the actual files that were created!

**URL:** http://localhost:9001/browser/warehouse/demo/employees  
**Credentials:** admin / password

MinIO is an S3-compatible object storage where all data and metadata files are stored.

**What to explore:**
- Click into the `data/` folder to see the actual Parquet data files
- Click into the `metadata/` folder to see Iceberg Metadata Json, Manifest and Manifest List files

## Congrats!

This concludes your first exercise. Now, you've learned to set up your Open Lakehouse by configuring a catalog and storage object and executing queries on them with engines like Trino and Apache Spark.